# Day 07 - Warp-specialized asynchronous pipelines

This lesson develops the producer/consumer progression from official
[Notebook 09](../notebooks/09_async_pipeline.ipynb), then turns it
into a five-step challenge. Two warps exchange values through a
shared-memory ring buffer:

```text
producer warp -> shared-memory stages -> consumer warp
```

The goal is not merely to call `PipelineAsync`. You should be able
to explain who owns each stage, which event changes its state, why a
stage cannot be overwritten early, and why the producer must drain
the protocol before retiring.

In [ ]:
import torch

import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack

cutlass.cuda.initialize_cuda_context()

## 1. Why two CTA barriers are correct but restrictive

A synchronous one-slot exchange can use this sequence:

```text
producer writes slot
     |
     +-- CTA barrier: value is now visible
     |
consumer reads slot
     |
     +-- CTA barrier: slot may be overwritten
```

Both barriers involve the entire CTA. Even when warp 0 only
produces and warp 1 only consumes, both warps repeatedly enter
lockstep. A pipeline replaces global rendezvous with stage-local
ownership transitions. That permits the producer to prepare a
future stage while the consumer works on an older one.

`PipelineAsync` coordinates readiness; the payload still lives in
shared memory. The pipeline does not move data for you.

## 2. The four transitions of one stage

Every reusable stage follows the same state machine:

| Actor | Operation | State transition | Meaning |
| --- | --- | --- | --- |
| Producer | `acquire_and_advance()` | empty -> producer-owned | Wait until this slot is reusable |
| Producer | `handle.commit()` | producer-owned -> full | Publish the completed write |
| Consumer | `wait_and_advance()` | full -> consumer-owned | Wait until the value is ready |
| Consumer | `handle.release()` | consumer-owned -> empty | Permit the next overwrite |

Acquire and wait are ordering points. Commit and release are
notifications. Omitting any one transition can produce stale data,
an overwrite race, or a deadlock.

In [ ]:
# Trace the indices used by a three-stage circular buffer.
stages = 3
items = 8
producer_slots = [item % stages for item in range(items)]
consumer_slots = [item % stages for item in range(items)]

assert producer_slots == consumer_slots
for item, slot in enumerate(producer_slots):
    print(f"item {item}: acquire/commit/wait/release stage {slot}")

## 3. Why multiple stages enable overlap

With one stage, the producer cannot begin item `i+1` until the
consumer releases item `i`. With three stages, it can publish items
0, 1, and 2 before wrapping back to stage 0. On wraparound,
`acquire_and_advance()` supplies backpressure: it waits until the
consumer has released that stage.

More stages are not automatically faster. Each stage consumes
shared memory and barrier state. Choose enough stages to cover the
latency gap between producer and consumer, but not so many that
shared-memory use reduces occupancy.

In this small exercise one Float32 is stored per stage. In GEMM,
each stage usually contains complete A and B tiles, so stage count
has a much larger resource cost.

In [ ]:
def stage_storage_bytes(stages, elements_per_stage, element_bytes=2):
    payload = stages * elements_per_stage * element_bytes
    barrier_words = stages * 2
    barrier_bytes = barrier_words * 8
    return payload, barrier_bytes


for stage_count in (1, 2, 3, 4):
    payload, barriers = stage_storage_bytes(
        stage_count, elements_per_stage=4096
    )
    print(
        f"{stage_count} stages: payload={payload / 1024:.1f} KiB, "
        f"barriers={barriers} B"
    )

## 4. CuTe objects that implement the protocol

Two cooperative groups describe the participant counts:

```python
producer_group = cutlass.pipeline.CooperativeGroup(
    cutlass.pipeline.Agent.Thread, 32
)
consumer_group = cutlass.pipeline.CooperativeGroup(
    cutlass.pipeline.Agent.Thread, 32
)
```

They describe one 32-thread producer warp and one 32-thread
consumer warp. `PipelineAsync.create` combines those groups with:

- `num_stages`, the circular-buffer depth;
- shared barrier storage, with two 64-bit words per stage;
- participant state returned by `make_participants()`.

The stage index comes from the acquired or waited handle. Always
use `handle.index` to address the matching payload slot; do not
maintain an unrelated manual modulo counter.

## 5. Why `producer.tail()` matters

Kernel completion does not replace protocol completion. The
producer can finish issuing commits while the consumer still owns
stages. `producer.tail()` waits for the expected final consumer
arrivals before the producer warp retires and the CTA's shared
storage disappears.

A useful rule:

```text
commit publishes one item;
tail drains all outstanding producer obligations.
```

Tail belongs after the producer loop, not after each item. Calling
it per item removes the overlap the pipeline was intended to
provide.

## 6. Challenge: build the staged exchange

Fill the TODOs in order:

1. Describe one producer and one consumer cooperative group.
2. Create a `PipelineAsync` over the shared barriers.
3. Acquire, write, and commit each producer item.
4. Drain the producer with `tail()`.
5. Wait, read, and release each consumer item.

The result must be `0..length-1` even when `length` exceeds the
stage count many times. That wraparound is the important case.

In [ ]:
@cute.kernel
def pipeline_kernel(
    shared_storage: cutlass.Constexpr,
    result: cute.Tensor,
    observed_stages: cute.Tensor,
):
    stages = cute.size(observed_stages)
    warp_idx = cute.arch.make_warp_uniform(cute.arch.warp_idx())

    smem = cutlass.utils.SmemAllocator()
    storage = smem.allocate(shared_storage, 64)
    stage_buffer = storage.stage_buffer.get_tensor(observed_stages.layout)
    stage_buffer.fill(0)
    cute.arch.sync_threads()

    # TODO(1): describe one producer warp and one consumer warp.
    # producer_group = cutlass.pipeline.CooperativeGroup(
    #     cutlass.pipeline.Agent.Thread, 32
    # )
    # consumer_group = ...
    raise NotImplementedError("Day 07 TODO(1): create cooperative groups")

    # TODO(2): create PipelineAsync with `stages` slots and the shared barriers.
    # pipeline = cutlass.pipeline.PipelineAsync.create(
    #     num_stages=stages,
    #     producer_group=producer_group,
    #     consumer_group=consumer_group,
    #     barrier_storage=storage.barriers.data_ptr(),
    # )
    # producer, consumer = pipeline.make_participants()

    if warp_idx == 0:
        for i in cutlass.range(cute.size(result)):
            # TODO(3): acquire a free slot, write `i`, then commit the slot.
            pass
        # TODO(4): call producer.tail() before the producer warp retires.

    if warp_idx == 1:
        for i in cutlass.range(cute.size(result)):
            # TODO(5): wait for a full slot, consume it, then release it.
            pass

    tidx, _, _ = cute.arch.thread_idx()
    if tidx == 0:
        observed_stages.store(stage_buffer.load())


@cute.jit
def run_pipeline(result: cute.Tensor, observed_stages: cute.Tensor):
    stages = cute.size(observed_stages)

    @cute.struct
    class SharedStorage:
        barriers: cute.struct.MemRange[cutlass.Int64, stages * 2]
        stage_buffer: cute.struct.Align[
            cute.struct.MemRange[cutlass.Float32, stages], 1024
        ]

    pipeline_kernel(SharedStorage, result, observed_stages).launch(
        grid=(1, 1, 1), block=(64, 1, 1)
    )


def run(length: int, stages: int) -> None:
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("Day 07 needs a CUDA device.")
    cutlass.cuda.initialize_cuda_context()
    result = torch.full((length,), -1, device="cuda", dtype=torch.float32)
    observed_stages = torch.zeros((stages,), device="cuda", dtype=torch.float32)
    run_pipeline(from_dlpack(result), from_dlpack(observed_stages))
    torch.cuda.synchronize()
    torch.testing.assert_close(
        result, torch.arange(length, device="cuda", dtype=torch.float32)
    )

## 7. Correctness experiment

Run after replacing all five TODOs. Try a stage count that does not
divide the item count, ensuring the ring wraps at different points.

In [ ]:
run(length=17, stages=3)
print("Pipeline preserved all 17 values across stage wraparound.")

## 8. Debugging a pipeline

| Symptom | Likely protocol error |
| --- | --- |
| Kernel hangs | Missing `commit`, `release`, or mismatched participant counts |
| Repeated/stale values | Consumer read before the matching wait |
| Values disappear after wraparound | Producer did not acquire before overwrite |
| Only stage 0 changes | Payload indexed manually instead of with `handle.index` |
| Final iterations hang | Producer omitted `tail()` or loop trip counts differ |

Pipeline bugs often appear as hangs rather than Python exceptions.
First compare producer and consumer trip counts. Then verify each
acquired handle is committed exactly once and each waited handle is
released exactly once.

## 9. Connection to a pipelined GEMM

Replace the scalar payload with A/B tiles and the roles become:

```text
Producer: acquire -> global-to-shared copy -> commit
Consumer: wait -> shared-to-register copy + MMA -> release
```

Day 10 initially uses one synchronous shared-memory stage so the
dataflow remains visible. The optimization path is to allocate
multiple A/B stages, prefetch future K tiles, and use this same
state machine to overlap movement with MMA.

**Checkpoint:** explain why `release()` must occur after the final
read or MMA that consumes the stage, not immediately after `wait()`.